# ASSIGNMENT GenAI – AI Resume Screening System with Tracing

## Data Science Internship – February 2026

### Objective
Build an AI-powered Resume Screening System using:
- LangChain
- LangSmith
- OpenAI API
- LCEL pipeline

### Output
Resume → Skills → Match → Score → Explanation


In [68]:
!pip install -q transformers langchain-core langsmith

In [69]:
import os
from transformers import pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "APIKEYHERE"
os.environ["LANGCHAIN_PROJECT"] = "AI Resume Screening Internship"

In [71]:
generator = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=200
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [72]:
def llm_invoke(prompt_value):
    prompt_text = prompt_value.to_string()

    result = generator(prompt_text)

    return result[0]["generated_text"]

In [73]:
llm_chain = RunnableLambda(llm_invoke)

In [74]:
job_description = """
Role: Data Scientist

Required Skills:
Python
Machine Learning
SQL
NLP
Deep Learning
Pandas
Scikit-learn
LangChain

Experience:
2+ years

Tools:
Jupyter, Git, PyTorch
"""

print(job_description)


Role: Data Scientist

Required Skills:
Python
Machine Learning
SQL
NLP
Deep Learning
Pandas
Scikit-learn
LangChain

Experience:
2+ years

Tools:
Jupyter, Git, PyTorch



In [75]:
strong_resume = """
John Doe
3 years Data Scientist experience
Skills: Python, SQL, NLP, Deep Learning, PyTorch, LangChain, Pandas
Worked on AI chatbot and screening systems
"""

average_resume = """
Jane Smith
1 year analyst experience
Skills: Python, SQL, Pandas, basic ML
Worked on dashboard reporting
"""

weak_resume = """
Alex Brown
Fresher
Skills: Excel, MS Word, communication
"""

In [76]:
extract_prompt = PromptTemplate.from_template("""
Extract only explicitly mentioned:
1. Skills
2. Experience
3. Tools

Do not assume anything missing.

Resume:
{resume}
""")

In [77]:
match_prompt = PromptTemplate.from_template("""
Compare the extracted resume details with the job description.

Job Description:
{jd}

Resume Extracted Data:
{resume_data}

Return:
- Matching skills
- Missing skills
- Experience fit
""")

In [78]:
score_prompt = PromptTemplate.from_template("""
Based on job requirements and resume match analysis,
assign a fit score from 0 to 100.

Scoring rules:
- Strong match: 80–100
- Average match: 50–79
- Weak match: 0–49

Return ONLY:
Score: XX
Reason: explanation
""")

In [79]:
extract_chain = extract_prompt | llm_chain
match_chain = match_prompt | llm_chain
score_chain = score_prompt | llm_chain

In [80]:
def screening_pipeline(resume):
    extracted = extract_chain.invoke({
        "resume": resume
    })

    matched = match_chain.invoke({
        "jd": job_description,
        "resume_data": extracted
    })

    score = score_chain.invoke({
        "match": matched
    })

    return {
        "extracted": extracted,
        "matched": matched,
        "score": score
    }

In [81]:
strong_result = screening_pipeline(strong_resume)

print("STRONG CANDIDATE")
print(strong_result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STRONG CANDIDATE
{'extracted': '\nExtract only explicitly mentioned:\n1. Skills\n2. Experience\n3. Tools\n\nDo not assume anything missing.\n\nResume:\n\nJohn Doe\n3 years Data Scientist experience\nSkills: Python, SQL, NLP, Deep Learning, PyTorch, LangChain, Pandas\nWorked on AI chatbot and screening systems\n\n\nThis is a job posting by a professional AI program manager. Please do not post about your work as a professional.\n\n\nPlease read the following for more information on how to proceed. Your resume should be complete, with a brief description of your background and qualifications as well as the title and contact email for the position.\n\nJob Description: Resume:\n\nJob Description:\n\nJob Description:\n\nJob Description: Resume: Resume:\n\nJob Description: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume:

In [82]:
average_result = screening_pipeline(average_resume)

print("AVERAGE CANDIDATE")
print(average_result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AVERAGE CANDIDATE
{'extracted': "\nExtract only explicitly mentioned:\n1. Skills\n2. Experience\n3. Tools\n\nDo not assume anything missing.\n\nResume:\n\nJane Smith\n1 year analyst experience\nSkills: Python, SQL, Pandas, basic ML\nWorked on dashboard reporting\n\n\nDATE:\n\nMay 2017\n\nTIME:\n\n5:00 AM - 5:30 PM\n\nLOCATION:\n\nSydney's Soho Club, 3221 N. State St\n\nNew Yorkers, NY 10026\n\nContact:\n\nJuan R. Saffron, Ph.D.,\n\nFinance Management Specialist\n\n[email protected]\n\nPhone: (212) 726-0943\n\nEmail: juan.saffron@gmail.com\n\nWebsite: https://www.saffron.com/\n\nLocation: Saffron's office on 23rd Street\n\nPhone: 604-622-4531\n\nWebsite: https://www.saffron.com/\n\nDescription:\n\nA comprehensive, and accurate summary of all aspects of the Saffron business. This is the only comprehensive and accurate version of our Saffron newsletter. It contains", 'matched': "\nCompare the extracted resume details with the job description.\n\nJob Description:\n\nRole: Data Scientist\n\

In [83]:
weak_result = screening_pipeline(weak_resume)

print("WEAK CANDIDATE")
print(weak_result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WEAK CANDIDATE
{'extracted': '\nExtract only explicitly mentioned:\n1. Skills\n2. Experience\n3. Tools\n\nDo not assume anything missing.\n\nResume:\n\nAlex Brown\nFresher\nSkills: Excel, MS Word, communication\n\n\nWhat are the different skills for each of these skills?\n\n1. Language Skills\n\n2. Writing\n\n3. Writing Tips\n\n4. Writing Tips for Business\n\n5. Writing Tips for Personal Growth\n\n6. Writing Tips for Growth Success\n\n7. Writing Tips for Successful Success\n\n8. Writing Tips for Successful Success\n\n9. Writing Tips for Successful Success\n\n10. Writing Tips for Successful Success\n\n11. Writing Tips for Successful Success\n\n12. Writing Tips for Successful Success\n\n13. Writing Tips for Successful Success\n\n14. Writing Tips for Successful Success\n\n15. Writing Tips for Successful Success\n\n16. Writing Tips for Successful Success\n\n17. Writing Tips for Successful Success\n\n18. Writing Tips for Successful Success\n\n19. Writing Tips for Successful Success\n\n20. W

In [87]:
debug_resume = """
Candidate knows Python only
"""

debug_result = screening_pipeline(debug_resume)

print(debug_result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'extracted': '\nExtract only explicitly mentioned:\n1. Skills\n2. Experience\n3. Tools\n\nDo not assume anything missing.\n\nResume:\n\nCandidate knows Python only\n\n\nCandidate is an experienced programmer who has developed a lot of useful software, but has not written a single codebase. He works in a small company, like Microsoft, but has never written anything that was well thought out. So he is not an expert in Python.\n\n\nNo one knows what he is working on, but he is working on something amazing. For example, he is working on a novel feature that will allow you to create a web app that is completely self-contained.\n\n\nI understand that you need to understand that this is just a demo. This is not a proof.\n\n\nThere is no problem with the fact as it is. It is just a demo. You are free to test the app. No one will ever use it.\n\n\nYou could ask him to show you how to make it working, but I think it is silly to ask a guy who is not an expert in Python to show you how to make a 

In [88]:
print("FINAL COMPARISON")
print("Strong:", strong_result["score"])
print("Average:", average_result["score"])
print("Weak:", weak_result["score"])

FINAL COMPARISON
Strong: 
Based on job requirements and resume match analysis,
assign a fit score from 0 to 100.

Scoring rules:
- Strong match: 80–100
- Average match: 50–79
- Weak match: 0–49

Return ONLY:
Score: XX
Reason: explanation

What do you think?
Average: 
Based on job requirements and resume match analysis,
assign a fit score from 0 to 100.

Scoring rules:
- Strong match: 80–100
- Average match: 50–79
- Weak match: 0–49

Return ONLY:
Score: XX
Reason: explanation

Scores and scores based on job requirements and resume match analysis.

The reason for the return is to make sure you have the best job offer.

The following are the required qualifications for the job.

The current job is a part-time position.

The current job is an independent contractor.

The current job is a part-time position.

The current job needs to work for at least six months in order to qualify.

The current job has a minimum salary of $100,000.

The current job has an annual benefit of at least $500,00

In [89]:
final_results = {
    "strong_candidate": strong_result,
    "average_candidate": average_result,
    "weak_candidate": weak_result
}

print(final_results)

{'strong_candidate': {'extracted': '\nExtract only explicitly mentioned:\n1. Skills\n2. Experience\n3. Tools\n\nDo not assume anything missing.\n\nResume:\n\nJohn Doe\n3 years Data Scientist experience\nSkills: Python, SQL, NLP, Deep Learning, PyTorch, LangChain, Pandas\nWorked on AI chatbot and screening systems\n\n\nThis is a job posting by a professional AI program manager. Please do not post about your work as a professional.\n\n\nPlease read the following for more information on how to proceed. Your resume should be complete, with a brief description of your background and qualifications as well as the title and contact email for the position.\n\nJob Description: Resume:\n\nJob Description:\n\nJob Description:\n\nJob Description: Resume: Resume:\n\nJob Description: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Resume: Res

# Observations and Insights

## Strong Candidate
High skill alignment with job description.
Strong experience fit.

## Average Candidate
Partial skill match.
Lacks NLP and Deep Learning.

## Weak Candidate
Low technical skill match.
No relevant experience.

## Challenges
- Prompt engineering
- Avoiding hallucinated assumptions
- Skill extraction consistency
- Explainable scoring